In [4]:
### Dependencies
# Base Dependencies
import os
import pickle
import sys

# LinAlg / Stats / Plotting Dependencies
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import umap.plot as umapplot
import umap.umap_ as umap

# Torch Dependencies
import torch
import torch.multiprocessing
import torchvision
import torch.utils.data.dataset as Dataset
from torchvision import transforms

# from pl_bolts.models.self_supervised import resnets
# from pl_bolts.utils.semi_supervised import Identity
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
torch.multiprocessing.set_sharing_strategy("file_system")

In [5]:
def create_UMAP(library_path, save_path, enc_name, n=50, d=0.5):
    path = os.path.join(library_path, '%s.pkl' % (enc_name))
    with open(path, 'rb') as handle:
        asset_dict = pickle.load(handle)
        embeddings, labels = asset_dict['embeddings'], asset_dict['labels']
    
    mapper = umap.UMAP(n_neighbors=n, min_dist=d).fit(embeddings)
    
    fig = plt.figure(figsize=(10, 10), dpi=100)
    
    # Define the colors for the labels
    unique_labels = np.unique(labels)
    colors = ['red', 'blue', 'green', 'yellow', 'black', 'orange', 'purple', 'gray', 'brown']
    label_color_map = dict(zip(unique_labels, colors))
    
    # Create color list for plotting
    label_colors = [label_color_map[label] for label in labels]
    
    # Plot with custom colors
    scatter = umapplot.points(mapper, labels=labels, color_key=label_color_map, width=600, height=600)
    
    plt.tight_layout()
    os.makedirs(os.path.join(save_path, 'UMAPs'), exist_ok=True)
    plt.savefig(os.path.join(save_path, 'UMAPs', '%s_umap_n%d_d%0.2f.jpg' % (enc_name, n, d)))
    plt.close(fig)

In [ ]:
library_path = './embeddings_patch_library/'
umap_save_path = './'
os.makedirs(umap_save_path, exist_ok=True)
models = ['dino', 'resnet50', 'dinocrc100k', 'resnet50crc100k']

for enc_name in models:
    create_UMAP(library_path=library_path, save_path=umap_save_path, enc_name=enc_name)
